# Land Use Change Comparison

Compares **one baseline land-use map** against **any number of land-use change scenarios**
(S1, S2, S3, ...). Nothing here assumes what the scenarios are -- green or grey measures, planning
alternatives, future projections -- they are just names you choose in the config cell.

**Per comparison** (Steps 3-7, run once for every pair in `COMPARE`):

1. a change map -- changed cells red, unchanged grey;
2. the full from -> to transition table, ranked;
3. a two-ring donut -- baseline (inner) vs scenario (outer);
4. a top-N transitions map, plus a QGIS `.qml` style so it opens already coloured.

**Across all maps at once** (Steps 8-12):

5. composition bars for every map;
6. the concentric donut -- **one ring per map**, baseline innermost;
7. a single-map donut, a CORINE reference legend, and the transition legends side by side.

Every class has **one colour everywhere** -- the official CORINE QGIS palette in the config
cell drives every figure *and* every `.qml` style.

In each comparison the "to" raster sets the reference grid; the other is aligned onto it with
a nearest-neighbour `WarpedVRT` (codes are categorical -- never interpolated). Everything is
read block by block, so raster size does not matter.

In [ ]:
import os
import sys

import csv
import textwrap
from collections import Counter
from pathlib import Path
from xml.sax.saxutils import escape

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib.patches import Patch
import rasterio
from rasterio.warp import Resampling
from rasterio.vrt import WarpedVRT
from pyproj import Geod
from pyproj import CRS as PyprojCRS

## Config

Three blocks. **Block 1 is the only one you normally edit**: the baseline map, the scenario
maps, and which comparisons to run. Adding a scenario is one line in `SCENARIO_MAPS` -- every
title, legend, ring and filename picks it up.

In [ ]:
# =======================================================================================
#  1. THE MAPS -- one baseline, any number of land-use change scenarios
# =======================================================================================
_EX = r"path/to/LU_map_folder"

BASELINE = "BASELINE"          # the baseline's name on every figure and in every filename
BASELINE_MAP = _EX + r"path/to/baseline_lu.tif"

# One entry per land-use change scenario -- as many as you have. The names are free text and
# appear exactly as written on the figures, so name them the way you want them read.
SCENARIO_MAPS = {
    "S1": _EX + r"path\to\scenario_1.tif",
    "S2": _EX + r"path\to\scenario_2.tif",
    # "S3": r"path\to\scenario_3.tif",
}

# Which comparisons Steps 3-7 run (each one writes its own maps, tables and donut):
#   "all"              every scenario against the baseline
#   ["S2"]             only these scenarios, each against the baseline
#   [("S1", "S2")]     scenario against scenario, as ("from", "to")
#   Plain names and pairs can be mixed: ["S1", ("S1", "S2")]
COMPARE = "all"

CORINE_CSV = _EX + r"path\to\corineorlanduse.csv"
CORINE_CODE_COL = "corine" #based on corine, this can be changed.
CORINE_LABEL_COL = "description"
CORINE_MANNING_COL = "N"       # column holding Manning's n, shown in the .qml labels; None = omit

OUTPUT_DIR = Path(_EX + "path\to\output_folder")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =======================================================================================
#  2. COLOURS -- one palette for every figure and every QGIS style
# =======================================================================================
# Official CORINE colours, straight from the QGIS .qml: (hex, description, Manning's n).
# A code missing here still gets a fixed colour of its own (see Step 1) -- add it here if you
# want it to match QGIS.
CORINE_QGIS = {
    112: ("#d7191c", "Discontinuous urban fabric", 0.011),
    121: ("#e13e2d", "Industrial or commercial units and public facilities", 0.011),
    132: ("#ea643f", "Dump sites", 0.011),
    141: ("#f48950", "Green urban areas", 0.150),
    142: ("#fdae61", "Sport and leisure facilities", 0.150),
    211: ("#fec279", "Non-irrigated arable land", 0.200),
    231: ("#fed790", "Pastures meadows and other permanent grasslands under agricultural use", 0.150),
    242: ("#ffeba8", "Complex cultivation patterns", 0.200),
    243: ("#ffffbf", "Land principally occupied by agriculture with significant areas of natural vegetation", 0.440),
    311: ("#eaf7b8", "Broad-leaved forest", 0.600),
    312: ("#d5eeb2", "Coniferous forest", 0.400),
    313: ("#c0e6ab", "Mixed forest", 0.500),
    321: ("#abdda4", "Natural grassland", 0.150),
    322: ("#8bc7aa", "Moors and heathland", 0.240),
    411: ("#6bb0af", "Inland marshes", 0.150),
    412: ("#4b9ab4", "Peatbogs", 0.150),
    512: ("#2b83ba", "Water bodies", 0.010),
}
OTHER_COLOR = "#9e9e9e"        # the grouped "Other" wedge / bar, everywhere

# Top-N transitions map: one colour per rank (1st, 2nd, ... most frequent conversion).
TRANSITION_COLORS = ["#e6194B", "#3cb44b", "#4363d8", "#f58231"]   # red, green, blue, orange
# Optional per-scenario override -- used by the map, its .qml AND the legend in Step 12, so the
# three can never disagree. Leave empty to use TRANSITION_COLORS for every scenario.
TRANSITION_COLORS_BY_SCENARIO = {
    # "S1": ["#8B1A1A", "#1E7B32", "#4363D8", "#E8E038"],   # maroon, green, blue, yellow
}

# =======================================================================================
#  3. SETTINGS
# =======================================================================================
CODE_NODATA = 0
TOP_N_TABLE = 15               # transitions printed per comparison (the CSV has all of them)
TOP_N_MAP = 4                  # transitions that get their own colour on the map
PIE_MIN_PCT_LABEL = 2.0        # two-ring donut: classes below this share -> "Other"
DONUT_AUTOPCT_MIN_PCT = 3.0    # two-ring donut: no on-wedge % below this share
PREVIEW_DOWNSAMPLE = 10

# ---- derived from the above -- nothing to edit below this line -----------------------
if BASELINE in SCENARIO_MAPS:
    raise ValueError(f"'{BASELINE}' is used both as the baseline and as a scenario name.")
LU_MAPS = {BASELINE: BASELINE_MAP, **SCENARIO_MAPS}


def _resolve_pairs(compare):
    items = list(SCENARIO_MAPS) if compare == "all" else list(compare)
    pairs = []
    for it in items:
        base, scen = (BASELINE, it) if isinstance(it, str) else tuple(it)
        for k in (base, scen):
            if k not in LU_MAPS:
                raise KeyError(f"COMPARE names '{k}', which is not a map. "
                               f"Available: {list(LU_MAPS)}")
        if base == scen:
            raise ValueError(f"COMPARE compares '{base}' with itself.")
        pairs.append((base, scen))
    return pairs


PAIRS = _resolve_pairs(COMPARE)


def pair_tag(base, scen):
    """Every output file of a comparison carries this, so runs never overwrite each other."""
    return f"{base}_to_{scen}"


def transition_colors(scen):
    cols = TRANSITION_COLORS_BY_SCENARIO.get(scen, TRANSITION_COLORS)
    if len(cols) < TOP_N_MAP:
        raise ValueError(f"{len(cols)} transition colours for '{scen}' but TOP_N_MAP = "
                         f"{TOP_N_MAP} -- add colours or lower TOP_N_MAP.")
    return cols


print(f"Baseline : {BASELINE}  ({Path(BASELINE_MAP).name})")
for _k, _p in SCENARIO_MAPS.items():
    print(f"Scenario : {_k}  ({Path(_p).name})")
print(f"\n{len(PAIRS)} comparison(s) to run: " + ", ".join(f"{b} -> {s}" for b, s in PAIRS))

## Step 1 -- CORINE labels and one colour per class

Labels (and Manning's n, if the CSV has it) come from the CORINE mapping CSV. Colours come from
`CORINE_QGIS` in the config, so every figure and every `.qml` style below agrees with your QGIS
maps class for class. A code missing from `CORINE_QGIS` still gets a colour derived from the
code itself -- never from its position in this particular run -- so it stays the same no matter
which maps are compared.

In [ ]:
code_to_label, code_to_manning = {}, {}
with open(CORINE_CSV, newline="", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        _c = int(float(row[CORINE_CODE_COL]))
        code_to_label[_c] = row[CORINE_LABEL_COL].strip()
        if CORINE_MANNING_COL and row.get(CORINE_MANNING_COL) not in (None, ""):
            code_to_manning[_c] = float(row[CORINE_MANNING_COL])
print(f"Loaded {len(code_to_label)} class labels"
      + (f" and {len(code_to_manning)} Manning values." if code_to_manning else "."))


def label(code):
    return code_to_label.get(code, CORINE_QGIS.get(code, (None, f"code {code}"))[1])


_FALLBACK = []
for _cm in ("tab20", "tab20b", "tab20c"):
    _FALLBACK.extend(plt.get_cmap(_cm)(i) for i in range(20))


def get_color(code):
    """RGBA for a CORINE code -- the same everywhere, in every run."""
    if code in CORINE_QGIS:
        return mcolors.to_rgba(CORINE_QGIS[code][0])
    return _FALLBACK[int(code) % len(_FALLBACK)]


def rgba_to_hex(rgba):
    r, g, b = (int(round(255 * v)) for v in rgba[:3])
    return f"#{r:02x}{g:02x}{b:02x}"


_missing = sorted(c for c in code_to_label if c not in CORINE_QGIS)
if _missing:
    print(f"Not in CORINE_QGIS, using a fixed fallback colour: {_missing}")

## Step 2 -- Raster helpers

`open_pair(base, scen)` opens one comparison: the `scen` ("to") raster sets the grid, and the
`base` ("from") raster is wrapped in a nearest-neighbour `WarpedVRT` if it is not already on
that exact grid. Pixel area is measured geodesically when a raster is geographic (EPSG:4326),
because multiplying a transform in degrees gives square degrees, not km².

In [ ]:
def pixel_area_km2_from_transform(transform, crs, raster_shape):
    """True pixel area in km2. A projected grid is width x height; a GEOGRAPHIC grid's
    transform is in degrees, so one representative pixel (at the raster centre) is measured
    on the WGS84 ellipsoid instead."""
    if not PyprojCRS.from_user_input(crs).is_geographic:
        return abs(transform.a * transform.e) / 1e6
    geod = Geod(ellps="WGS84")
    px_w, px_h = abs(transform.a), abs(transform.e)
    h, w = raster_shape
    lon0, lat0 = transform * (w // 2, h // 2)
    area_m2, _ = geod.polygon_area_perimeter(
        [lon0, lon0 + px_w, lon0 + px_w, lon0], [lat0, lat0, lat0 - px_h, lat0 - px_h])
    return abs(area_m2) / 1e6


def read_code_block(src, window):
    arr = src.read(1, window=window, masked=True).filled(CODE_NODATA)
    arr = np.rint(arr).astype("int32")
    arr[arr < 0] = CODE_NODATA
    return arr


def open_pair(base, scen):
    """-> (scen_src, base_src, to_close, aligned). scen_src sets the grid."""
    scen_src = rasterio.open(LU_MAPS[scen])
    base_raw = rasterio.open(LU_MAPS[base])
    same = (base_raw.crs == scen_src.crs
            and base_raw.transform.almost_equals(scen_src.transform)
            and base_raw.width == scen_src.width and base_raw.height == scen_src.height)
    if same:
        return scen_src, base_raw, [scen_src, base_raw], False
    vrt = WarpedVRT(base_raw, crs=scen_src.crs, transform=scen_src.transform,
                    width=scen_src.width, height=scen_src.height,
                    resampling=Resampling.nearest)
    return scen_src, vrt, [vrt, base_raw, scen_src], True


def close_all(handles):
    for h in handles:
        h.close()


print("Helpers ready.")

## Step 3 -- Pass 1, every comparison: change map + transition counts + class totals

For each pair in `COMPARE`: writes the binary change map (1 = changed, 0 = unchanged,
255 = outside the domain of either map) and counts every from -> to transition and every
class in both maps. Everything later steps need is kept in `RESULTS`, keyed by
`(from, to)`.

In [ ]:
RESULTS = {}

for base, scen in PAIRS:
    tag = pair_tag(base, scen)
    print(f"\n=== {base} -> {scen} " + "=" * max(0, 60 - len(tag)))
    scen_src, base_src, _handles, aligned = open_pair(base, scen)
    print(f"  '{base}' {'aligned onto' if aligned else 'already on'} the '{scen}' grid"
          + (" (nearest-neighbour)" if aligned else ""))

    transform, crs = scen_src.transform, scen_src.crs
    width, height = scen_src.width, scen_src.height
    px_km2 = pixel_area_km2_from_transform(transform, crs, (height, width))
    _geo = PyprojCRS.from_user_input(crs).is_geographic
    print(f"  grid {width} x {height}, pixel {px_km2 * 1e6:,.1f} m2 "
          f"[{'geographic -- measured geodesically' if _geo else 'projected'}]")

    change_out = OUTPUT_DIR / f"luc_change_map_{tag}.tif"
    profile = dict(driver="GTiff", height=height, width=width, count=1, dtype="uint8",
                   crs=crs, transform=transform, nodata=255, compress="deflate",
                   tiled=True, blockxsize=256, blockysize=256, BIGTIFF="YES")

    transitions, counts_from, counts_to = Counter(), Counter(), Counter()
    n_valid = n_changed = 0
    with rasterio.open(change_out, "w", **profile) as dst:
        for _, window in scen_src.block_windows(1):
            from_b = read_code_block(base_src, window)
            to_b = read_code_block(scen_src, window)
            valid = (from_b != CODE_NODATA) & (to_b != CODE_NODATA)
            changed = valid & (from_b != to_b)

            out_b = np.full(from_b.shape, 255, dtype="uint8")
            out_b[valid] = 0
            out_b[changed] = 1
            dst.write(out_b, 1, window=window)

            if valid.any():
                _pairs, _n = np.unique(np.stack([from_b[valid], to_b[valid]], axis=1),
                                       axis=0, return_counts=True)
                for (f_, t_), c in zip(_pairs, _n):
                    transitions[(int(f_), int(t_))] += int(c)
                for arr, store in ((from_b, counts_from), (to_b, counts_to)):
                    _c, _k = np.unique(arr[valid], return_counts=True)
                    for c, k in zip(_c, _k):
                        store[int(c)] += int(k)
            n_valid += int(valid.sum())
            n_changed += int(changed.sum())
    close_all(_handles)

    pct_changed = 100 * n_changed / max(n_valid, 1)
    RESULTS[(base, scen)] = dict(
        tag=tag, base=base, scen=scen, transform=transform, crs=crs, width=width,
        height=height, px_km2=px_km2, transitions=transitions, counts_from=counts_from,
        counts_to=counts_to, n_valid=n_valid, n_changed=n_changed, pct_changed=pct_changed,
        change_out=change_out)
    print(f"  valid {n_valid * px_km2:,.2f} km2, changed {n_changed * px_km2:,.2f} km2 "
          f"-> {pct_changed:.2f}% of the domain changed class")
    print(f"  saved: {change_out.name}")

## Step 4 -- Ranked transition table, every comparison

No-change pairs (from == to) are excluded from the ranking. The full table, including
no-change rows, goes to `luc_transition_table_<from>_to_<to>.csv`.

In [ ]:
for (base, scen), r in RESULTS.items():
    px = r["px_km2"]
    changes_only = {k: v for k, v in r["transitions"].items() if k[0] != k[1]}
    r["ranked"] = sorted(changes_only.items(), key=lambda kv: kv[1], reverse=True)

    print(f"\n=== {base} -> {scen}: {len(r['ranked'])} distinct conversions ===")
    for (f_, t_), n in r["ranked"][:TOP_N_TABLE]:
        print(f"  {label(f_)[:44]:<45} -> {label(t_)[:44]:<45} "
              f"{n:>10,} cells  ({n * px:>8.2f} km2)")

    _csv = OUTPUT_DIR / f"luc_transition_table_{r['tag']}.csv"
    with open(_csv, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["from_code", "from_label", "to_code", "to_label", "n_cells", "area_km2",
                    "is_change"])
        for (fc, tc), n in sorted(r["transitions"].items(), key=lambda kv: kv[1], reverse=True):
            w.writerow([fc, label(fc), tc, label(tc), n, round(n * px, 4), fc != tc])
    print(f"  saved: {_csv.name}")
    if r["ranked"]:
        (f_, t_), n = r["ranked"][0]
        print(f"  largest single conversion: {label(f_)} -> {label(t_)} "
              f"({n * px:.2f} km2, {100 * n / max(r['n_changed'], 1):.1f}% of all change)")

## Step 5 -- Two-ring donut, every comparison

Inner ring = the "from" map, outer ring = the "to" map, same colour per class in both rings.
Classes below `PIE_MIN_PCT_LABEL` in a ring are grouped into a grey "Other"; the legend still
shows every kept class's share in both maps as `from% / to%`. For all maps in one figure, see
the concentric donut in Step 9.

In [ ]:
# --- sizing knobs ------------------------------------------------------------------------
DONUT_FIG_WIDTH, DONUT_FIG_HEIGHT = 15, 9
DONUT_AXES_WIDTH_FRAC = 0.42       # share of the canvas width the donut gets
DONUT_FONT_SCALE = 1.5             # scales every text size in this figure
INNER_RADIUS, INNER_WIDTH = 0.70, 0.42
OUTER_RADIUS, OUTER_WIDTH = 1.00, 0.28
# -----------------------------------------------------------------------------------------


def _ring(counts, total):
    full_pct = {c: 100 * n / total for c, n in counts.items()} if total else {}
    order = sorted(counts, key=lambda c: counts[c], reverse=True)
    kept = [c for c in order if full_pct[c] >= PIE_MIN_PCT_LABEL]
    dropped = [c for c in order if c not in kept]
    vals = [counts[c] for c in kept]
    cols = [get_color(c) for c in kept]
    if dropped:
        vals.append(sum(counts[c] for c in dropped))
        cols.append(mcolors.to_rgba(OTHER_COLOR))
    return vals, cols, kept, full_pct, dropped


def _autopct(p):
    return f"{p:.1f}%" if p >= DONUT_AUTOPCT_MIN_PCT else ""


for (base, scen), r in RESULTS.items():
    cf, ct = r["counts_from"], r["counts_to"]
    tot_f, tot_t = sum(cf.values()), sum(ct.values())

    _csv = OUTPUT_DIR / f"luc_composition_by_class_{r['tag']}.csv"
    with open(_csv, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["code", "label", f"{base}_cells", f"{base}_pct", f"{scen}_cells", f"{scen}_pct"])
        for c in sorted(set(cf) | set(ct)):
            n0, n1 = cf.get(c, 0), ct.get(c, 0)
            w.writerow([c, label(c), n0, round(100 * n0 / max(tot_f, 1), 3),
                        n1, round(100 * n1 / max(tot_t, 1), 3)])

    iv, ic, ik, ipct, idrop = _ring(cf, tot_f)
    ov, oc, ok, opct, odrop = _ring(ct, tot_t)

    _aw = DONUT_AXES_WIDTH_FRAC * DONUT_FIG_WIDTH
    _ah = min(0.92, _aw / DONUT_FIG_HEIGHT * 1.05)
    fig = plt.figure(figsize=(DONUT_FIG_WIDTH, DONUT_FIG_HEIGHT))
    ax = fig.add_axes([0.02, (1 - _ah) / 2, DONUT_AXES_WIDTH_FRAC, _ah])
    _fs = 9.5 * DONUT_FONT_SCALE
    _, _, it = ax.pie(iv, colors=ic, radius=INNER_RADIUS, startangle=90, counterclock=False,
                      wedgeprops=dict(width=INNER_WIDTH, edgecolor="white", linewidth=1),
                      autopct=_autopct, pctdistance=(INNER_RADIUS - INNER_WIDTH / 2) / INNER_RADIUS,
                      textprops=dict(fontsize=_fs))
    _, _, ot = ax.pie(ov, colors=oc, radius=OUTER_RADIUS, startangle=90, counterclock=False,
                      wedgeprops=dict(width=OUTER_WIDTH, edgecolor="white", linewidth=1),
                      autopct=_autopct, pctdistance=(OUTER_RADIUS - OUTER_WIDTH / 2) / OUTER_RADIUS,
                      textprops=dict(fontsize=_fs))
    for t in it + ot:
        t.set_path_effects([pe.withStroke(linewidth=2.5, foreground="white")])

    ax.text(0, 0, f"Inner:\n{base}\n\nOuter:\n{scen}", ha="center", va="center",
            fontsize=12 * DONUT_FONT_SCALE, fontweight="bold")
    ax.set_title(f"Land use composition -- {base} (inner) vs {scen} (outer)",
                 fontsize=15 * DONUT_FONT_SCALE)

    _codes = sorted(set(ik) | set(ok), key=lambda c: max(ipct.get(c, 0), opct.get(c, 0)),
                    reverse=True)
    handles = [Patch(facecolor=get_color(c), edgecolor="none",
                     label=f"{label(c)}  ({ipct.get(c, 0):.1f}% / {opct.get(c, 0):.1f}%)")
               for c in _codes]
    for name, drop, pcts in ((base, idrop, ipct), (scen, odrop, opct)):
        if drop:
            handles.append(Patch(facecolor=OTHER_COLOR, edgecolor="none",
                                 label=f"Other ({name}, {len(drop)} classes): "
                                       f"{sum(pcts[c] for c in drop):.1f}%"))
    fig.legend(handles=handles, loc="center left",
               bbox_to_anchor=(DONUT_AXES_WIDTH_FRAC + 0.04, 0.5),
               fontsize=11 * DONUT_FONT_SCALE, frameon=True,
               title=f"Class  ({base}% / {scen}%)", title_fontsize=11.5 * DONUT_FONT_SCALE,
               handlelength=1.6, handleheight=1.6, labelspacing=0.7)

    _png = OUTPUT_DIR / f"luc_composition_donut_{r['tag']}.png"
    plt.savefig(_png, dpi=180, bbox_inches="tight", pad_inches=0.3)
    plt.show()
    plt.close(fig)
    print(f"Saved: {_png.name}  and  {_csv.name}")

## Step 6 -- Pass 2, every comparison: top-N transitions map + its QGIS style

Each cell becomes 0 (other / unchanged) or the rank 1..N of its transition; 255 = nodata. A
`.qml` sidecar with the colours and full `from -> to (area)` labels is written next to the
tif, so QGIS opens it already styled. If the layer is already open in QGIS, remove and re-add
it -- QGIS only reads the sidecar at load time.

In [ ]:
def write_paletted_qml(qml_path, entries):
    """entries: [(value, '#hex', alpha 0-255, label)] -> a paletted-raster QGIS style."""
    palette = "\n".join(
        f'        <paletteEntry value="{v}" color="{c}" alpha="{a}" label="{escape(l)}"/>'
        for v, c, a, l in entries)
    qml = f"""<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>
<qgis version="3.34.0" styleCategories="AllStyleCategories">
  <pipe>
    <rasterrenderer type="paletted" band="1" opacity="1" alphaBand="-1" nodataColor="">
      <rasterTransparency/>
      <colorPalette>
{palette}
      </colorPalette>
    </rasterrenderer>
    <brightnesscontrast brightness="0" contrast="0" gamma="1"/>
    <huesaturation saturation="0" grayscaleMode="0" colorizeOn="0"/>
    <resamplingStage>resamplingFilter</resamplingStage>
  </pipe>
  <blendMode>0</blendMode>
</qgis>
"""
    with open(qml_path, "w", encoding="utf-8") as fh:
        fh.write(qml)


for (base, scen), r in RESULTS.items():
    px = r["px_km2"]
    top = r["ranked"][:TOP_N_MAP]
    r["top"] = top
    cols = transition_colors(scen)
    rank_of = {pair: i + 1 for i, (pair, _) in enumerate(top)}

    print(f"\n=== {base} -> {scen}: top {len(top)} transitions ===")
    for i, ((f_, t_), n) in enumerate(top, start=1):
        print(f"  {i}. {label(f_)} -> {label(t_)}  ({n * px:.2f} km2)")

    topmap_out = OUTPUT_DIR / f"luc_top_transitions_map_{r['tag']}.tif"
    profile = dict(driver="GTiff", height=r["height"], width=r["width"], count=1,
                   dtype="uint8", crs=r["crs"], transform=r["transform"], nodata=255,
                   compress="deflate", tiled=True, blockxsize=256, blockysize=256,
                   BIGTIFF="YES")
    scen_src, base_src, _handles, _ = open_pair(base, scen)
    with rasterio.open(topmap_out, "w", **profile) as dst:
        for _, window in scen_src.block_windows(1):
            from_b = read_code_block(base_src, window)
            to_b = read_code_block(scen_src, window)
            valid = (from_b != CODE_NODATA) & (to_b != CODE_NODATA)
            out_b = np.full(from_b.shape, 255, dtype="uint8")
            out_b[valid] = 0
            for (f_, t_), rank in rank_of.items():
                out_b[valid & (from_b == f_) & (to_b == t_)] = rank
            dst.write(out_b, 1, window=window)
    close_all(_handles)
    r["topmap_out"] = topmap_out

    entries = [(0, "#bfbfbf", 90, "Other / unchanged")]
    entries += [(i, cols[i - 1], 255, f"{label(f_)} -> {label(t_)} ({n * px:.1f} km2)")
                for i, ((f_, t_), n) in enumerate(top, start=1)]
    _qml = str(topmap_out).rsplit(".tif", 1)[0] + ".qml"
    write_paletted_qml(_qml, entries)
    print(f"  saved: {topmap_out.name}  +  {Path(_qml).name}")

## Step 6c -- QGIS styles for the land-use maps themselves

Every map in `LU_MAPS` gets a `.qml` listing each CORINE code present in it, labelled
`"112 - Discontinuous urban fabric (n=0.011)"` and coloured with the same palette as every
figure here. If a raster sits on a read-only drive, set `LUC_QML_DIR` to a writable folder and
load the style manually (Properties -> Symbology -> Load Style).

In [ ]:
LUC_QML_FOR = list(LU_MAPS)    # which maps get a style
LUC_QML_DIR = None             # None = next to each raster; or e.g. OUTPUT_DIR


def codes_present(path):
    found = set()
    with rasterio.open(path) as src:
        for _, window in src.block_windows(1):
            arr = src.read(1, window=window, masked=True).filled(CODE_NODATA)
            found.update(np.unique(np.rint(arr).astype("int32")).tolist())
    found.discard(CODE_NODATA)
    return sorted(c for c in found if c >= 0)


for name in LUC_QML_FOR:
    path = LU_MAPS[name]
    rows = []
    for c in codes_present(path):
        n = code_to_manning.get(c)
        lab = f"{c} - {label(c)}" + (f" (n={n:.3f})" if n is not None else "")
        rows.append((c, rgba_to_hex(get_color(c)), 255, lab))
    stem = Path(path).stem
    qml_path = (Path(LUC_QML_DIR) / f"{stem}.qml") if LUC_QML_DIR else \
        Path(str(path).rsplit(".tif", 1)[0] + ".qml")
    try:
        write_paletted_qml(qml_path, rows)
        print(f"{name}: {len(rows)} classes -> {qml_path}")
    except OSError as e:
        print(f"{name}: could not write style ({e}). Set LUC_QML_DIR to a writable folder.")

## Step 7 -- Preview plots, every comparison: change map + top-N transitions map

In [ ]:
def decimated_read(path, small_h, small_w):
    with rasterio.open(path) as src:
        return src.read(1, out_shape=(small_h, small_w), resampling=Resampling.nearest)


for (base, scen), r in RESULTS.items():
    px, tag = r["px_km2"], r["tag"]
    origin = "upper" if r["transform"].e < 0 else "lower"
    sh = max(1, r["height"] // PREVIEW_DOWNSAMPLE)
    sw = max(1, r["width"] // PREVIEW_DOWNSAMPLE)

    # --- change map ---
    small = decimated_read(r["change_out"], sh, sw)
    rgba = np.zeros((*small.shape, 4), dtype="float32")
    rgba[small == 0] = (0.6, 0.6, 0.6, 0.5)
    rgba[small == 1] = (0.86, 0.08, 0.24, 1.0)
    fig, ax = plt.subplots(figsize=(9, 8))
    ax.imshow(rgba, origin=origin)
    ax.set_title(f"Land use change, {base} -> {scen} [preview, 1/{PREVIEW_DOWNSAMPLE}]")
    ax.axis("off")
    ax.legend(handles=[
        plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=(0.6, 0.6, 0.6),
                   markersize=14, label="Unchanged"),
        plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=(0.86, 0.08, 0.24),
                   markersize=14, label="Changed")],
        loc="upper left", bbox_to_anchor=(1.02, 1), frameon=True)
    ax.text(0.02, 0.02, f"{r['pct_changed']:.2f}% of domain changed\n"
                        f"({r['n_changed'] * px:.2f} km2 of {r['n_valid'] * px:.2f} km2)",
            transform=ax.transAxes, va="bottom", ha="left", fontsize=10,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85))
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"luc_change_map_preview_{tag}.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    # --- top-N transitions map ---
    cols = transition_colors(scen)
    small = decimated_read(r["topmap_out"], sh, sw)
    rgba = np.zeros((*small.shape, 4), dtype="float32")
    rgba[small == 0] = (0.75, 0.75, 0.75, 0.35)
    for i in range(1, len(r["top"]) + 1):
        rgba[small == i] = mcolors.to_rgba(cols[i - 1])
    fig, ax = plt.subplots(figsize=(9, 8))
    ax.imshow(rgba, origin=origin)
    ax.set_title(f"Top {len(r['top'])} land use conversions, {base} -> {scen} "
                 f"[preview, 1/{PREVIEW_DOWNSAMPLE}]")
    ax.axis("off")
    handles = [plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=(0.75, 0.75, 0.75),
                          markersize=14, label="Other / unchanged")]
    handles += [plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=cols[i - 1],
                           markersize=14,
                           label=f"{label(f_)} -> {label(t_)} ({n * px:.1f} km2)")
                for i, ((f_, t_), n) in enumerate(r["top"], start=1)]
    ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.02, 1), frameon=True,
              fontsize=8)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"luc_top_transitions_preview_{tag}.png", dpi=200,
                bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("Previews done.")

## Step 8 -- Composition across all maps (counts + stacked bars)

Aligns every map in `COMPOSITION_MAPS` onto one reference grid and counts classes only over
cells valid in **all** of them, so the percentages are strictly comparable. Defines `pct`,
`counts`, `all_codes_seen` and `n_common_valid` -- Steps 9 and 10 read these, so this cell
has to run first.

In [ ]:
# ---- config -----------------------------------------------------------------------------
COMPOSITION_MAPS = "all"      # "all" = baseline + every scenario; or a list of names
COMPOSITION_REF = None        # whose grid to align onto; None = the first map in the list
BAR_MIN_PCT = 1.0             # classes below this share in EVERY map are folded into "Other"
BAR_LABEL_MIN_PCT = 2.0       # only annotate segments at least this wide
MAP_BAR_COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860",
                  "#DA8BC3", "#8C8C8C"]
# -----------------------------------------------------------------------------------------

COMPOSITION_MAPS = list(LU_MAPS) if COMPOSITION_MAPS == "all" else list(COMPOSITION_MAPS)
_unknown = [m for m in COMPOSITION_MAPS if m not in LU_MAPS]
if _unknown:
    raise KeyError(f"COMPOSITION_MAPS names {_unknown}; available: {list(LU_MAPS)}")
ref_name = COMPOSITION_REF or COMPOSITION_MAPS[0]
print(f"Maps: {COMPOSITION_MAPS}\nReference grid: '{ref_name}' -- only cells valid in ALL "
      f"maps are counted, so the percentages are strictly comparable.")

ref_raw = rasterio.open(LU_MAPS[ref_name])
_rt, _rc, _rw, _rh = ref_raw.transform, ref_raw.crs, ref_raw.width, ref_raw.height
comp_pixel_area_km2 = pixel_area_km2_from_transform(_rt, _rc, (_rh, _rw))

opened, to_close = {}, [ref_raw]
for name in COMPOSITION_MAPS:
    raw = ref_raw if name == ref_name else rasterio.open(LU_MAPS[name])
    if raw is not ref_raw:
        to_close.append(raw)
    if (raw.crs == _rc and raw.transform.almost_equals(_rt)
            and raw.width == _rw and raw.height == _rh):
        opened[name] = raw
    else:
        opened[name] = WarpedVRT(raw, crs=_rc, transform=_rt, width=_rw, height=_rh,
                                 resampling=Resampling.nearest)
        to_close.append(opened[name])
        print(f"  '{name}' aligned onto the '{ref_name}' grid (nearest-neighbour)")

counts = {m: Counter() for m in COMPOSITION_MAPS}
n_common_valid = 0
for _, window in ref_raw.block_windows(1):
    blocks = {m: read_code_block(src, window) for m, src in opened.items()}
    valid = np.ones(next(iter(blocks.values())).shape, dtype=bool)
    for arr in blocks.values():
        valid &= arr != CODE_NODATA
    if not valid.any():
        continue
    n_common_valid += int(valid.sum())
    for m, arr in blocks.items():
        cs, ns = np.unique(arr[valid], return_counts=True)
        for c, n in zip(cs, ns):
            counts[m][int(c)] += int(n)
close_all(to_close)

domain_txt = f"common domain {n_common_valid * comp_pixel_area_km2:,.0f} km2"
print(f"\nCommon valid domain: {n_common_valid:,} cells ({domain_txt[14:]})")

pct = {m: {c: 100 * n / max(n_common_valid, 1) for c, n in counts[m].items()}
       for m in COMPOSITION_MAPS}
all_codes_seen = sorted(set().union(*[set(p) for p in pct.values()]))
kept = [c for c in all_codes_seen if any(pct[m].get(c, 0) >= BAR_MIN_PCT for m in COMPOSITION_MAPS)]
kept.sort(key=lambda c: -max(pct[m].get(c, 0) for m in COMPOSITION_MAPS))
dropped = [c for c in all_codes_seen if c not in kept]
cats = [f"{c}\n{label(c)}" for c in kept]
if dropped:
    cats.append(f"Other\n({len(dropped)} classes <{BAR_MIN_PCT}%)")

y = np.arange(len(cats))
fig, ax = plt.subplots(figsize=(14, max(6, 1.15 * len(cats))))
left = np.zeros(len(cats))
for i, m in enumerate(COMPOSITION_MAPS):
    vals = np.array([pct[m].get(c, 0) for c in kept]
                    + ([sum(pct[m].get(c, 0) for c in dropped)] if dropped else []))
    ax.barh(y, vals, 0.65, left=left, label=m, color=MAP_BAR_COLORS[i % len(MAP_BAR_COLORS)])
    for yi, (l, v) in enumerate(zip(left, vals)):
        if v >= BAR_LABEL_MIN_PCT:
            ax.text(l + v / 2, yi, f"{v:.1f}", ha="center", va="center", fontsize=12,
                    fontweight="bold", color="white")
    left += vals
ax.set_yticks(y)
ax.set_yticklabels(cats, fontsize=13)
ax.invert_yaxis()
ax.set_xlabel("Share of common valid domain [%]  (one segment per map)", fontsize=14)
ax.set_title(f"Land-use composition by class, all maps ({domain_txt})", fontsize=16)
ax.tick_params(axis="x", labelsize=12)
ax.legend(title="Map", fontsize=13, title_fontsize=14, loc="lower right")
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "luc_composition_all_maps.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

with open(OUTPUT_DIR / "luc_composition_all_maps.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["code", "label"] + [f"{m}_pct" for m in COMPOSITION_MAPS]
               + [f"{m}_km2" for m in COMPOSITION_MAPS])
    for c in all_codes_seen:
        w.writerow([c, label(c)] + [round(pct[m].get(c, 0), 3) for m in COMPOSITION_MAPS]
                   + [round(counts[m].get(c, 0) * comp_pixel_area_km2, 4) for m in COMPOSITION_MAPS])
print("Saved: luc_composition_all_maps.png / .csv")

## Step 9 -- Concentric donut: one ring per map

**The big all-in-one donut.** One ring per map in `COMPOSITION_MAPS`, the baseline innermost,
the scenarios outward in the order they are listed. Ring names sit outside the chart on leader
lines whose angles are chosen automatically to avoid crossing the % labels. Reads `pct` and
`all_codes_seen` from Step 8.

In [ ]:
# ---------------- STYLE CONFIG -- edit freely ----------------
FONT_FAMILY     = "Times New Roman"
FS_TITLE        = 22
FS_WEDGE        = 18                  # the % numbers inside the rings
FS_RING_LABEL   = 18                  # map names outside the chart
FS_LEGEND       = 14
FS_LEGEND_TITLE = 16

PCT_DECIMALS    = 1
PCT_SUFFIX      = "%"
PCT_OUTER_ONLY  = False               # True -> "%" only on the outermost ring
PCT_HALO        = True
WEDGE_TEXT_COLOR = "black"            # "auto" = black/white per wedge luminance

# two SEPARATE thresholds -- do not merge them
DONUT_MIN_PCT_LABEL = 5.0             # wedges under this get no printed number (cosmetic)
DONUT_MIN_PCT_KEEP  = 2.0             # classes under this in EVERY map -> "Other" (data)

DONUT_RING_WIDTH    = 0.32
DONUT_INNER_R       = 0.45
DONUT_LEGEND_NCOL   = 2
WEDGE_ORDER         = "code"          # "code" (palette reads as a gradient) | "size"
SHOW_MANNING        = False
DONUT_SHOW_LEGEND   = False
DONUT_FIGSIZE       = (14, 13)
DONUT_DPI           = 300

RING_LABEL_AUTO     = True            # pick spoke angles that dodge the % labels
RING_LABEL_WINDOW   = (96, 160)       # arc the spokes may use (deg; 90 = straight up)
RING_SPOKE_MIN_SEP  = 11              # min degrees between spokes
RING_LABEL_GAP      = 0.22
RING_LABEL_PAD      = 0.04
RING_LINE_COLOR     = "0.20"
RING_LINE_WIDTH     = 1.3
RING_DOT_SIZE       = 5
AXIS_PAD            = 1.55
# -------------------------------------------------------------

_avail = {f.name for f in mpl.font_manager.fontManager.ttflist}
mpl.rcParams["font.family"] = (FONT_FAMILY if FONT_FAMILY in _avail else next(
    (a for a in ("Liberation Serif", "DejaVu Serif", "Nimbus Roman") if a in _avail), "serif"))


def _readable(hexcol):
    if WEDGE_TEXT_COLOR != "auto":
        return WEDGE_TEXT_COLOR
    r_, g_, b_ = mcolors.to_rgb(hexcol)
    return "black" if (0.2126 * r_ + 0.7152 * g_ + 0.0722 * b_) > 0.55 else "white"


def _angdist(a, b):
    d = abs((a - b) % 360.0)
    return min(d, 360.0 - d)


def _class_label(code):
    n = code_to_manning.get(code, CORINE_QGIS.get(code, (None, None, None))[2])
    return f"{code} - {label(code)}" + (f" (n={n:.3f})" if SHOW_MANNING and n is not None else "")


donut_kept = [c for c in all_codes_seen
              if any(pct[m].get(c, 0) >= DONUT_MIN_PCT_KEEP for m in COMPOSITION_MAPS)]
donut_kept.sort() if WEDGE_ORDER == "code" else donut_kept.sort(
    key=lambda c: -max(pct[m].get(c, 0) for m in COMPOSITION_MAPS))
donut_dropped = [c for c in all_codes_seen if c not in donut_kept]

wedge_colors = [rgba_to_hex(get_color(c)) for c in donut_kept]
wedge_labels = [_class_label(c) for c in donut_kept]
if donut_dropped:
    wedge_colors.append(OTHER_COLOR)
    wedge_labels.append(f"Other ({len(donut_dropped)} classes <{DONUT_MIN_PCT_KEEP:g}%)")

_n = len(COMPOSITION_MAPS)
r_max = DONUT_INNER_R + _n * DONUT_RING_WIDTH

# ---- pass 1: the rings --------------------------------------------------------------------
fig, ax = plt.subplots(figsize=DONUT_FIGSIZE)
label_angles_per_ring = []
for i, m in enumerate(COMPOSITION_MAPS):
    vals = [pct[m].get(c, 0) for c in donut_kept]
    if donut_dropped:
        vals.append(sum(pct[m].get(c, 0) for c in donut_dropped))
    _sfx = PCT_SUFFIX if (not PCT_OUTER_ONLY or i == _n - 1) else ""
    r_out = DONUT_INNER_R + (i + 1) * DONUT_RING_WIDTH
    wedges, _, autotexts = ax.pie(
        vals, radius=r_out, colors=wedge_colors, startangle=90, counterclock=False,
        wedgeprops=dict(width=DONUT_RING_WIDTH, edgecolor="white", linewidth=1.4),
        autopct=(lambda p, _s=_sfx: f"{p:.{PCT_DECIMALS}f}{_s}" if p >= DONUT_MIN_PCT_LABEL else ""),
        pctdistance=1 - (DONUT_RING_WIDTH / 2) / r_out,
        textprops=dict(fontsize=FS_WEDGE, fontweight="bold"))
    angs = []
    for w, t, col in zip(wedges, autotexts, wedge_colors):
        tc = _readable(col)
        t.set_color(tc)
        if not t.get_text():
            continue
        angs.append(0.5 * (w.theta1 + w.theta2))
        if PCT_HALO:
            t.set_zorder(20)
            t.set_path_effects([pe.withStroke(linewidth=3.0,
                                              foreground="white" if tc != "white" else "black")])
    label_angles_per_ring.append(angs)

# ---- pass 2: leader lines that dodge every % label they cross -----------------------------
_lo, _hi = RING_LABEL_WINDOW
if RING_LABEL_AUTO:
    _cand = np.linspace(_lo, _hi, 361)
    _angles, _worst = [], []
    for i in range(_n):
        blocked = [a for j in range(i, _n) for a in label_angles_per_ring[j]]
        best, best_score = _cand[0], -1e9
        for c in _cand:
            s = min((_angdist(c, b) for b in blocked), default=180.0)
            s -= 1000.0 * sum(_angdist(c, pc) < RING_SPOKE_MIN_SEP for pc in _angles)
            if s > best_score:
                best_score, best = s, c
        _angles.append(best)
        _worst.append(best_score)
    if min(_worst) < 6:
        print(f"!! tightest spoke clears a number by only {min(_worst):.1f} deg -- "
              f"widen RING_LABEL_WINDOW or raise DONUT_MIN_PCT_LABEL")
else:
    _angles = np.linspace(_lo, _hi, _n) if _n > 1 else np.array([_lo])

for i, m in enumerate(COMPOSITION_MAPS):
    r_out = DONUT_INNER_R + (i + 1) * DONUT_RING_WIDTH
    th = np.deg2rad(_angles[i])
    ct, st = np.cos(th), np.sin(th)
    r_mid, r_end = r_out - DONUT_RING_WIDTH / 2, r_max + RING_LABEL_GAP
    ax.plot([r_mid * ct, r_end * ct], [r_mid * st, r_end * st], color=RING_LINE_COLOR,
            lw=RING_LINE_WIDTH, zorder=15, solid_capstyle="round",
            path_effects=[pe.withStroke(linewidth=RING_LINE_WIDTH + 2.2, foreground="white")])
    ax.plot([r_mid * ct], [r_mid * st], marker="o", ms=RING_DOT_SIZE, color=RING_LINE_COLOR,
            zorder=16, path_effects=[pe.withStroke(linewidth=2.2, foreground="white")])
    ax.text((r_end + RING_LABEL_PAD) * ct, (r_end + RING_LABEL_PAD) * st, m,
            ha="left" if ct >= 0 else "right", va="center", fontsize=FS_RING_LABEL,
            fontweight="bold", color="black", zorder=17)

ax.set_xlim(-r_max * AXIS_PAD, r_max * AXIS_PAD)
ax.set_ylim(-r_max * AXIS_PAD, r_max * AXIS_PAD)
ax.set_aspect("equal")
ax.set_title("Land-use composition per map\n"
             f"(rings inner to outer: {'  ->  '.join(COMPOSITION_MAPS)};  {domain_txt})",
             fontsize=FS_TITLE, pad=22)

if DONUT_SHOW_LEGEND:
    leg = ax.legend(handles=[Patch(facecolor=c, edgecolor="0.5", linewidth=0.5, label=l)
                             for c, l in zip(wedge_colors, wedge_labels)],
                    loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=DONUT_LEGEND_NCOL,
                    fontsize=FS_LEGEND, frameon=True, title="Land-use class",
                    title_fontsize=FS_LEGEND_TITLE, handlelength=1.8, columnspacing=1.4,
                    labelspacing=0.7)
    leg.get_title().set_fontweight("bold")

plt.tight_layout()
for _ext in ("png", "pdf"):
    _fn = OUTPUT_DIR / f"luc_composition_donut_all_maps.{_ext}"
    plt.savefig(_fn, dpi=DONUT_DPI, bbox_inches="tight", facecolor="white")
    print(f"Saved: {_fn.name}")
plt.show()
plt.close(fig)
print("spoke angles (deg): " + ", ".join(f"{m}={a:.0f}" for m, a in zip(COMPOSITION_MAPS, _angles)))

print(f"\n{'code':>6}  {'class':<62}" + "".join(f"{m:>11}" for m in COMPOSITION_MAPS))
print("-" * (70 + 11 * _n))
for c in donut_kept:
    print(f"{c:>6}  {label(c)[:60]:<62}" + "".join(f"{pct[m].get(c, 0):>10.1f}%" for m in COMPOSITION_MAPS))
if donut_dropped:
    print(f"{'':>6}  {f'Other ({len(donut_dropped)} classes)':<62}"
          + "".join(f"{sum(pct[m].get(c, 0) for c in donut_dropped):>10.1f}%" for m in COMPOSITION_MAPS))

## Step 10 -- Single-map donut

One ring for one map. Narrow wedges get their label outside on a short leader line.

In [ ]:
# ---------------- CONFIG -- edit freely ----------------
RING_MAP        = BASELINE            # which map to draw -- any name in COMPOSITION_MAPS
RING_TITLE      = None                # None -> "Land-use composition -- <map>"
FS_TITLE_1      = 30
FS_WEDGE_1      = 25
AXIS_PAD_1      = 1.20                # blank canvas around the ring
TITLE_PAD       = 2
TITLE_Y         = None
PCT_DECIMALS_1  = 1
RING_MIN_PCT_LABEL = 2.0
RING_MIN_PCT_KEEP  = 2.0
RING_ORDER      = "code"              # "code" | "size"
LABEL_MIN_DEG   = 14                  # wedges narrower than this -> label outside
LABEL_OUT_R     = 1.14
RING_WIDTH_FRAC = 0.42
RING_FIGSIZE    = (9, 9)
RING_DPI        = 300
# --------------------------------------------------------

if RING_MAP not in pct:
    raise KeyError(f"RING_MAP '{RING_MAP}' is not in COMPOSITION_MAPS {COMPOSITION_MAPS}")

_keep = [c for c in all_codes_seen if pct[RING_MAP].get(c, 0) >= RING_MIN_PCT_KEEP]
_keep.sort() if RING_ORDER == "code" else _keep.sort(key=lambda c: -pct[RING_MAP].get(c, 0))
_drop = [c for c in all_codes_seen if c not in _keep]
_vals = [pct[RING_MAP].get(c, 0) for c in _keep]
_cols = [rgba_to_hex(get_color(c)) for c in _keep]
if _drop:
    _vals.append(sum(pct[RING_MAP].get(c, 0) for c in _drop))
    _cols.append(OTHER_COLOR)

fig, ax = plt.subplots(figsize=RING_FIGSIZE)
wedges, _, autotexts = ax.pie(
    _vals, colors=_cols, radius=1.0, startangle=90, counterclock=False,
    wedgeprops=dict(width=RING_WIDTH_FRAC, edgecolor="white", linewidth=2.0),
    autopct=lambda p: f"{p:.{PCT_DECIMALS_1}f}%" if p >= RING_MIN_PCT_LABEL else "",
    pctdistance=1 - RING_WIDTH_FRAC / 2, textprops=dict(fontsize=FS_WEDGE_1, fontweight="bold"))
for w, t, col in zip(wedges, autotexts, _cols):
    if not t.get_text():
        continue
    if (w.theta2 - w.theta1) < LABEL_MIN_DEG:
        a = np.deg2rad(0.5 * (w.theta1 + w.theta2))
        cx, cy = np.cos(a), np.sin(a)
        t.set_position((LABEL_OUT_R * cx, LABEL_OUT_R * cy))
        t.set_color("black")
        t.set_ha("left" if cx >= 0 else "right")
        t.set_va("center")
        ax.plot([1.005 * cx, (LABEL_OUT_R - 0.03) * cx], [1.005 * cy, (LABEL_OUT_R - 0.03) * cy],
                color="0.35", lw=1.0, zorder=5, solid_capstyle="round")
    else:
        r_, g_, b_ = mcolors.to_rgb(col)
        t.set_color("black" if (0.2126 * r_ + 0.7152 * g_ + 0.0722 * b_) > 0.55 else "white")

ax.set_xlim(-AXIS_PAD_1, AXIS_PAD_1)
ax.set_ylim(-AXIS_PAD_1, AXIS_PAD_1)
ax.set_aspect("equal")
_tkw = dict(fontsize=FS_TITLE_1, fontweight="bold", pad=TITLE_PAD)
if TITLE_Y is not None:
    _tkw["y"] = TITLE_Y
ax.set_title(RING_TITLE or f"Land-use composition -- {RING_MAP}", **_tkw)
plt.tight_layout()
for _ext in ("png", "pdf"):
    _fn = OUTPUT_DIR / f"luc_donut_{RING_MAP}.{_ext}"
    plt.savefig(_fn, dpi=RING_DPI, bbox_inches="tight", facecolor="white")
    print(f"Saved: {_fn.name}")
plt.show()
plt.close(fig)

print(f"\n{RING_MAP}")
for c in _keep:
    print(f"  {c:>4}  {label(c)[:58]:<60} {pct[RING_MAP].get(c, 0):>6.1f}%")
if _drop:
    print(f"  {'':>4}  {f'Other ({len(_drop)} classes)':<60} "
          f"{sum(pct[RING_MAP].get(c, 0) for c in _drop):>6.1f}%")

## Step 11 -- CORINE reference legend (standalone image)

Every class in `CORINE_QGIS` (or only those present in the maps), bold caps, wrapped labels,
bordered swatches, plus the grey "Other" entry matching the donuts.

In [ ]:
# ---------------- LEGEND CONFIG -- edit freely ----------------
LEG_TITLE       = "CORINE LAND COVER CLASSES"   # None for no title
LEG_CLASSES     = "all"      # "all" = every class in CORINE_QGIS; "present" = only in your maps
LEG_MIN_PCT     = 2.0        # only used with "present"
LEG_ADD_OTHER   = True
LEG_OTHER_LABEL = "Other (still within CORINE LUC)"
LEG_NCOL        = 3
LEG_WRAP        = 26
LEG_UPPERCASE   = True
LEG_BOLD        = True
LEG_FONTSIZE    = 13
LEG_TITLE_SIZE  = 17
LEG_SWATCH      = 2.0
LEG_SWATCH_EDGE = "black"
LEG_SWATCH_LW   = 1.0
LEG_ROWSPACE    = 1.15
LEG_COLSPACE    = 2.4
LEG_HANDLEPAD   = 1.0
LEG_SHOW_MANNING = False
LEG_FRAME       = True
LEG_FRAME_COLOR = "0.55"
LEG_TRANSPARENT = False
LEG_WIDTH_IN    = 13
LEG_ROW_IN      = 0.50       # inches per TEXT LINE
LEG_NAME        = "corine_legend"
LEG_DPI         = 300
# --------------------------------------------------------------

if LEG_CLASSES == "present":
    _codes = sorted(c for c in all_codes_seen
                    if any(pct[m].get(c, 0) >= LEG_MIN_PCT for m in COMPOSITION_MAPS))
else:
    _codes = sorted(CORINE_QGIS)


def _wrap_leg(txt):
    return textwrap.fill(txt.upper() if LEG_UPPERCASE else txt, width=LEG_WRAP)


def _fmt(code):
    n = code_to_manning.get(code, CORINE_QGIS.get(code, (None, None, None))[2])
    return _wrap_leg(f"{code} - {label(code)}"
                     + (f" (n={n:.3f})" if LEG_SHOW_MANNING and n is not None else ""))


_labs = [_fmt(c) for c in _codes]
_lcols = [rgba_to_hex(get_color(c)) for c in _codes]
if LEG_ADD_OTHER:
    _labs.append(_wrap_leg(LEG_OTHER_LABEL))
    _lcols.append(OTHER_COLOR)

_h = [Patch(facecolor=c, edgecolor=LEG_SWATCH_EDGE, linewidth=LEG_SWATCH_LW, label=l)
      for c, l in zip(_lcols, _labs)]
_rows = -(-len(_h) // LEG_NCOL)
_lines = max((sum(l.count("\n") + 1 for l in _labs[i * _rows:(i + 1) * _rows])
              for i in range(LEG_NCOL) if _labs[i * _rows:(i + 1) * _rows]), default=1)

fig = plt.figure(figsize=(LEG_WIDTH_IN, LEG_ROW_IN * _lines + (1.1 if LEG_TITLE else 0.5)))
leg = fig.legend(handles=_h, loc="center", ncol=LEG_NCOL,
                 prop={"size": LEG_FONTSIZE, "weight": "bold" if LEG_BOLD else "normal"},
                 frameon=LEG_FRAME, title=LEG_TITLE, handlelength=LEG_SWATCH,
                 handleheight=LEG_SWATCH, handletextpad=LEG_HANDLEPAD,
                 labelspacing=LEG_ROWSPACE, columnspacing=LEG_COLSPACE, borderpad=1.0)
if LEG_TITLE:
    leg.get_title().set_fontsize(LEG_TITLE_SIZE)
    leg.get_title().set_fontweight("bold")
if LEG_FRAME:
    leg.get_frame().set_edgecolor(LEG_FRAME_COLOR)
    leg.get_frame().set_facecolor("none" if LEG_TRANSPARENT else "white")
if LEG_TRANSPARENT:
    fig.patch.set_alpha(0)
plt.axis("off")
for _ext in ("png", "pdf"):
    _fn = OUTPUT_DIR / f"{LEG_NAME}.{_ext}"
    plt.savefig(_fn, dpi=LEG_DPI, bbox_inches="tight", transparent=LEG_TRANSPARENT,
                facecolor="none" if LEG_TRANSPARENT else "white")
    print(f"Saved: {_fn.name}")
plt.show()
plt.close(fig)

## Step 12 -- Transition legends, one column per comparison

Built from `RESULTS` in memory -- the same counts and the same `transition_colors()` the maps in
Step 6 used -- so each legend always matches its own map. No CSV is re-read and no raster is
opened again.

In [ ]:
# ---------------- CONFIG -- edit freely ----------------
TL_PAIRS        = "all"      # "all" = every comparison run above; or e.g. [("BASELINE", "S1")]
TL_OTHER_COLOR  = "#bfbfbf"
TL_OTHER_LABEL  = "Other / unchanged"
TL_SHOW_AREA    = True
TL_AREA_FMT     = "{:.1f}\u00a0km$^2$"      # non-breaking space: number and unit never split
PANEL_TITLE_FMT = "Top {n} LUC changes: {base} to {scen}"
TL_FONTSIZE     = 17
TL_TITLE_SIZE   = 20
TL_WRAP         = 40
TL_BOLD         = False
TL_TITLE_BOLD   = True
TL_SWATCH       = 1.5
TL_SWATCH_EDGE  = "black"
TL_SWATCH_LW    = 0.8
TL_ROWSPACE     = 0.95
TL_HANDLEPAD    = 0.8
TL_BORDERPAD    = 0.8
TL_FRAME        = True
TL_FRAME_COLOR  = "0.55"
TL_TRANSPARENT  = False
TL_PANEL_W_IN   = 7.0
TL_ROW_IN       = 0.44
TL_WSPACE       = 0.0        # 0.0 = boxes touching; negative to overlap
TL_EQUALISE     = True       # pad shorter columns so every box is the same height
TL_NAME         = "luc_transition_legends"
TL_DPI          = 300
TL_SAVE_SINGLES = True
# --------------------------------------------------------

_tl_pairs = list(RESULTS) if TL_PAIRS == "all" else [tuple(p) for p in TL_PAIRS]
_panels = []
for base, scen in _tl_pairs:
    if (base, scen) not in RESULTS:
        print(f"!! {base} -> {scen} was not run in Steps 3-6 -- skipped")
        continue
    r = RESULTS[(base, scen)]
    cols = transition_colors(scen)
    rows = [(TL_OTHER_COLOR, textwrap.fill(TL_OTHER_LABEL, TL_WRAP))]
    for i, ((f_, t_), n) in enumerate(r["top"]):
        lab = f"{label(f_)} -> {label(t_)}"
        if TL_SHOW_AREA:
            lab += "  (" + TL_AREA_FMT.format(n * r["px_km2"]) + ")"
        rows.append((cols[i], textwrap.fill(lab, TL_WRAP)))
    _panels.append((base, scen, rows))
if not _panels:
    raise RuntimeError("Nothing to draw -- run Steps 3-6 first.")


def _nlines(rows):
    return sum(l.count("\n") + 1 for _, l in rows)


_tl_lines = max(_nlines(rows) for _, _, rows in _panels)
if TL_EQUALISE:
    for _, _, rows in _panels:
        while _nlines(rows) < _tl_lines:
            rows.append(("none", " "))
_tl_height = TL_ROW_IN * _tl_lines + 1.6


def _tl_handles(rows):
    return [Patch(facecolor=c, edgecolor="none" if c == "none" else TL_SWATCH_EDGE,
                  linewidth=TL_SWATCH_LW, label=l) for c, l in rows]


def _tl_style(leg):
    leg.get_title().set_fontsize(TL_TITLE_SIZE)
    if TL_TITLE_BOLD:
        leg.get_title().set_fontweight("bold")
    if TL_FRAME:
        leg.get_frame().set_edgecolor(TL_FRAME_COLOR)
        leg.get_frame().set_facecolor("none" if TL_TRANSPARENT else "white")


_tl_kw = dict(prop={"size": TL_FONTSIZE, "weight": "bold" if TL_BOLD else "normal"},
              frameon=TL_FRAME, handlelength=TL_SWATCH, handleheight=TL_SWATCH,
              handletextpad=TL_HANDLEPAD, labelspacing=TL_ROWSPACE, borderpad=TL_BORDERPAD)

fig, axes = plt.subplots(1, len(_panels), figsize=(TL_PANEL_W_IN * len(_panels), _tl_height))
axes = [axes] if len(_panels) == 1 else list(axes)
for ax, (base, scen, rows) in zip(axes, _panels):
    leg = ax.legend(handles=_tl_handles(rows), loc="upper left",
                    bbox_to_anchor=(0.0, 0.0, 1.0, 1.0), mode="expand",
                    title=PANEL_TITLE_FMT.format(n=TOP_N_MAP, base=base, scen=scen), **_tl_kw)
    _tl_style(leg)
    ax.axis("off")
if TL_TRANSPARENT:
    fig.patch.set_alpha(0)
plt.tight_layout()
fig.subplots_adjust(wspace=TL_WSPACE)       # after tight_layout, which sets its own
for _ext in ("png", "pdf"):
    _fn = OUTPUT_DIR / f"{TL_NAME}.{_ext}"
    plt.savefig(_fn, dpi=TL_DPI, bbox_inches="tight", transparent=TL_TRANSPARENT,
                facecolor="none" if TL_TRANSPARENT else "white")
    print(f"Saved: {_fn.name}")
plt.show()
plt.close(fig)

if TL_SAVE_SINGLES:
    for base, scen, rows in _panels:
        f1 = plt.figure(figsize=(TL_PANEL_W_IN, _tl_height))
        _tl_style(f1.legend(handles=_tl_handles(rows), loc="center",
                            title=PANEL_TITLE_FMT.format(n=TOP_N_MAP, base=base, scen=scen),
                            **_tl_kw))
        if TL_TRANSPARENT:
            f1.patch.set_alpha(0)
        plt.axis("off")
        _fn = OUTPUT_DIR / f"{TL_NAME}_{pair_tag(base, scen)}.png"
        plt.savefig(_fn, dpi=TL_DPI, bbox_inches="tight", transparent=TL_TRANSPARENT,
                    facecolor="none" if TL_TRANSPARENT else "white")
        plt.close(f1)
        print(f"Saved: {_fn.name}")